<a href="https://colab.research.google.com/github/AkhilKas/edgeguard/blob/guardrail-classifier-training/edge_guard_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**EdgeGuard - Guardrail classifier training**

Runtime > Change runtime type > select a GPU (T4 is fine, free tier)

**Description:** This trains a small text classifier (DistilBERT) to tell apart
normal prompts from injection attempts, then exports it to ONNX so it can run on the Arduino UNO Q.


In [1]:
# ── Cell 1: Install dependencies ───────────────────────────────
!pip install transformers datasets torch scikit-learn onnx onnxruntime -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 30.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.


In [2]:
# ── Cell 2: Load the dataset

from datasets import load_dataset

# Starting dataset: clean, pre-labeled, no access request needed.
# label 0 = safe, label 1 = injection attempt
dataset = load_dataset("deepset/prompt-injections")

print(dataset)
print(dataset["train"][0])

README.md:   0%|          | 0.00/500 [00:00<?, ?B/s]

data/train-00000-of-00001-9564e8b05b4757(…): reconstructing file:   0%|          |  0.00B / 40.3kB            

data/train-00000-of-00001-9564e8b05b4757(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-701d16158af8736(…): reconstructing file:   0%|          |  0.00B / 10.9kB            

data/test-00000-of-00001-701d16158af8736(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/546 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/116 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 546
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 116
    })
})
{'text': 'Refugee crisis in Europe solutions', 'label': 0}


In [3]:
# ── Cell 3: Tokenize ─────────────────────────────────────────────────
# The model can't read raw text - it needs it converted to token IDs
# first (remember: tokens are the model's basic unit, like words/word-
# pieces). This step does that conversion for every example at once.

from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"  # small (66M params), good starting point
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized = dataset.map(tokenize_batch, batched=True)



config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/546 [00:00<?, ? examples/s]

Map:   0%|          | 0/116 [00:00<?, ? examples/s]

In [5]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"  # small (66M params), good starting point
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized = dataset.map(tokenize_batch, batched=True)


In [8]:
# ── Cell 4: Load the base model for classification

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,  # safe vs injection
)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
# ── Cell 5: Set up training
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

training_args = TrainingArguments(
    output_dir="./guardrail_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)
# deepset/prompt-injections only ships a "train" split by default in
# some versions - if there's no separate test split, carve one out:
if "test" not in tokenized:
    split = tokenized["train"].train_test_split(test_size=0.2, seed=42)
    tokenized["train"] = split["train"]
    tokenized["test"] = split["test"]

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    compute_metrics=compute_metrics,
)

In [12]:
# ── Cell 6: Train ─────────────────────────────────────────────────────
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.574341,0.369437,0.896552,1.000000,0.800000,0.888889
2,0.199486,0.236958,0.896552,1.000000,0.800000,0.888889
3,0.060716,0.249291,0.905172,1.000000,0.816667,0.899083


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=105, training_loss=0.2460442724682036, metrics={'train_runtime': 35.8471, 'train_samples_per_second': 45.694, 'train_steps_per_second': 2.929, 'total_flos': 54245399749632.0, 'train_loss': 0.2460442724682036, 'epoch': 3.0})

In [14]:
 #── Cell 7: Final evaluation ──────────────────────────────────────────
# This is the number you report in your write-up (section 6.2:
# precision, recall, F1 on held-out examples)
results = trainer.evaluate()
print(results)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.060716,0.249291,3,0.905172,1.000000,0.816667,0.899083


{'eval_loss': 0.2492905557155609, 'eval_accuracy': 0.9051724137931034, 'eval_precision': 1.0, 'eval_recall': 0.8166666666666667, 'eval_f1': 0.8990825688073395}


In [16]:
!pip install onnxscript -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 7.6 MB/s eta 0:00:00


In [18]:
# ── Cell 8: Export to ONNX ────────────────────────────────────────────
# This is the "compile the firmware" step - converts the trained
# PyTorch model into a portable format that runs on the board without
# needing PyTorch installed there.
import torch

model = model.to("cpu")
model.eval()
dummy_input = tokenizer("example text", return_tensors="pt", padding="max_length", max_length=128)

torch.onnx.export(
    model,
    (dummy_input["input_ids"], dummy_input["attention_mask"]),
    "guardrail_classifier.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size"},
        "attention_mask": {0: "batch_size"},
        "logits": {0: "batch_size"},
    },
    opset_version=14,
)

print("Exported to guardrail_classifier.onnx")

# Also save the tokenizer's vocabulary/config - the board needs this
# too, since it has to tokenize incoming text the same way the model
# was trained to expect.
tokenizer.save_pretrained("./guardrail_tokenizer")



/tmp/ipykernel_582/3062012478.py:11: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0927 04:51:12.040000 582 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `DistilBertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DistilBertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /project/onnx/version_converter/adapters/no_previous_version.h:23: adap

[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exported to guardrail_classifier.onnx


('./guardrail_tokenizer/tokenizer_config.json',
 './guardrail_tokenizer/tokenizer.json')

In [19]:
from google.colab import files
files.download("guardrail_classifier.onnx")

!zip -r guardrail_tokenizer.zip guardrail_tokenizer
files.download("guardrail_tokenizer.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  adding: guardrail_tokenizer/ (stored 0%)
  adding: guardrail_tokenizer/tokenizer_config.json (deflated 43%)
  adding: guardrail_tokenizer/tokenizer.json (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
!ls -la *.onnx*

-rw-r--r-- 1 root root    760052 Sep 27 04:51 guardrail_classifier.onnx
-rw-r--r-- 1 root root 267827200 Sep 27 04:51 guardrail_classifier.onnx.data


In [21]:
files.download("guardrail_classifier.onnx.data")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ── Cell 9: Download the files ────────────────────────────────────────
# In Colab, run this to download both to your laptop:
# from google.colab import files
# files.download("guardrail_classifier.onnx")
# !zip -r guardrail_tokenizer.zip guardrail_tokenizer
# files.download("guardrail_tokenizer.zip")
#
# From your laptop, scp both to the board:
#   scp guardrail_classifier.onnx arduino@<board-ip>:~/
#   scp guardrail_tokenizer.zip arduino@<board-ip>:~/